# Featuresmith Tutorial: 02 — Complete Dataset Review Walkthrough

Deep dive into Featuresmith's Review Engine — exploring the 8 automated reviewers, category filtering, finding severities, reviewer configuration, and formatted text report rendering.

---


## 1. Why Automated Dataset Code Reviews Matter
Just as software engineers perform code reviews before merging pull requests, ML engineers must perform dataset code reviews before training models. Featuresmith's Review Engine runs 8 specialized reviewers to evaluate dataset health deterministically.

### The 8 Automated Reviewers in v0.2.0
1. **Schema Health Reviewer**: Evaluates structural consistency and column naming.
2. **Data Types Reviewer**: Detects text types, numeric types, and type mismatches.
3. **Missing Values Reviewer**: Identifies column missingness ratios and null spikes.
4. **Duplicate Records Reviewer**: Checks for duplicate row entries.
5. **Constant Columns Reviewer**: Finds zero-variance and empty columns.
6. **High Cardinality Reviewer**: Flags categorical columns with excessive unique values.
7. **Basic Statistics Reviewer**: Analyzes distribution skewness and kurtosis anomalies.
8. **Leakage Risk Reviewer**: Evaluates target correlations, identifier shapes, timestamp anomalies, and duplicate targets.

### Prerequisite: Prepare the Sales Dataset
This notebook loads `examples/data/processed/sales.csv`, which `examples/prepare_datasets.py` generates deterministically (no network). From the repository root, run:

```bash
python examples/prepare_datasets.py
```


### Step 1: Load Dataset & Run Complete Review

In [1]:
import os

import featuresmith as fs

data_path = os.path.join("..", "data", "processed", "sales.csv")
dataset = fs.load(data_path)
review_res = fs.review(dataset)

print(f"Total Sections Evaluated : {len(review_res.sections)}")
print(f"Overall Summary          : {review_res.overall_summary}")

Total Sections Evaluated : 8
Overall Summary          : 4 of 8 sections passed with 5 finding(s) identified across the review.


### Step 2: Inspect Review Sections & Findings

In [2]:
for section in review_res.sections:
    sev_str = (
        section.severity.value
        if hasattr(section.severity, "value")
        else str(section.severity)
    )
    print(f"[{sev_str.upper():<8}] {section.title} ({len(section.findings)} findings)")
    for finding in section.findings:
        print(
            f"     - Column: {finding.column_name or 'dataset':<15} | {finding.title}"
        )

[CRITICAL] Schema Health (1 findings)
     - Column: return_reason   | Fully empty column 'return_reason'
[WARNING ] Constant Columns (1 findings)
     - Column: store_version   | Constant column 'store_version'
[WARNING ] Basic Statistics (2 findings)
     - Column: sales_amount    | High skewness in column 'sales_amount'
     - Column: sales_amount    | High kurtosis in column 'sales_amount'
[INFO    ] Data Types (1 findings)
     - Column: order_id        | Text column 'order_id'
[PASSED  ] Missing Values (0 findings)
[PASSED  ] Duplicate Rows (0 findings)
[PASSED  ] High Cardinality (0 findings)
[PASSED  ] Leakage Detection (0 findings)


### Step 3: Filter Review Categories & Configure Reviewers
You can pass `enabled_categories` as a list of the public `fs.ReviewCategory` enum members (e.g. `[fs.ReviewCategory.QUALITY, fs.ReviewCategory.LEAKAGE]`) or configure specific reviewer thresholds via `reviewer_config`.

Filtering to the `QUALITY` and `LEAKAGE` categories runs the 5 quality reviewers plus the leakage reviewer, for 6 sections total.

In [3]:
custom_review = fs.review(
    dataset,
    enabled_categories=[
        fs.ReviewCategory.QUALITY,
        fs.ReviewCategory.LEAKAGE,
    ],
    reviewer_config={"review.quality.missingness": {"threshold": 10.0}},
)
print(f"Filtered Review Sections Count: {len(custom_review.sections)}")

Filtered Review Sections Count: 6


### Step 4: Render Formatted Text Report with `fs.render()`

In [4]:
report_text = fs.render(review_res, target="console")
print("=== Formatted Text Report Preview ===")
print(report_text[:600] + "\n...")

=== Formatted Text Report Preview ===
Featuresmith Dataset Review
Rows: 1,000 | Columns: 10
Engine: v0.2.0
Generated: 2026-08-11T23:41:25.132973+00:00

4 of 8 sections passed with 5 finding(s) identified across the review.

[CRITICAL] Schema Health (review.schema.health)
  - Fully empty column 'return_reason' [return_reason]
      Column 'return_reason' contains only null/missing values.
[WARNING] Constant Columns (review.quality.constants)
  - Constant column 'store_version' [store_version]
      Column 'store_version' contains only one unique value (excluding nulls) and provides no variance.
[WARNING] Basic Statistics (review.qu
...


### Key Takeaways & Connection to Next Tutorial
- `fs.review()` evaluates dataset health across 8 reviewers deterministically.
- Category filtering (`enabled_categories`) and reviewer threshold overrides (`reviewer_config`) allow custom validation rules.
- `fs.render(review_res)` generates formatted console text reports.

**Next Tutorial**: In `03_ml_readiness_score.ipynb`, we explore the 0–100 ML Readiness Scorecard, health dimension weights, deduction math, and actionable fix suggestions.